In [1]:
# Cell 1. Install

!pip install -q -U "transformers==4.51.3" "tokenizers>=0.21,<0.22" accelerate safetensors scikit-learn pillow tqdm

In [16]:
# Cell 1. Imports / Config

import os, re, ast, random, itertools
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.backends.cuda.matmul.allow_tf32 = True

PROJECT_DIR = Path(r"C:\SNU_2026")
DATA_DIR = PROJECT_DIR / "data"

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV = DATA_DIR / "test.csv"
TRAIN_IMAGE_DIR = DATA_DIR / "train"
TEST_IMAGE_DIR = DATA_DIR / "test"

MODEL_ID = "google/siglip2-so400m-patch16-384"
MODEL_SAVE_DIR = PROJECT_DIR / "models" / "single_siglip2_high_score"
MODEL_SAVE_DIR.mkdir(parents=True, exist_ok=True)

BEST_PATH = MODEL_SAVE_DIR / "best_pairwise_ranker.pt"
OUTPUT_PATH = PROJECT_DIR / "submission_pairwise_ranker_safe.csv"

IMAGE_BATCH_SIZE = 16
TEXT_BATCH_SIZE = 128
BATCH_SIZE = 256

EPOCHS = 30
PATIENCE = 12
LR = 1e-4
WEIGHT_DECAY = 1e-3
NO_ORDERING_WEIGHT = 0.45

USE_EXISTING_CHECKPOINT = False  # 이미 best_pairwise_ranker.pt 있으면 바로 사용

print("device:", device)
print("BEST_PATH:", BEST_PATH)
print("checkpoint exists:", BEST_PATH.exists())

device: cuda
BEST_PATH: C:\SNU_2026\models\single_siglip2_high_score\best_pairwise_ranker.pt
checkpoint exists: True


In [17]:
# Cell 2. Data / Answer Utils

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

PERMS = list(itertools.permutations([1, 2, 3, 4]))
PERM_TO_CLASS = {p: i for i, p in enumerate(PERMS)}
CLASS_TO_PERM = {i: p for i, p in enumerate(PERMS)}
IDENTITY_CLASS = PERM_TO_CLASS[(1, 2, 3, 4)]
PAIR_INDEX = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]

def parse_answer(x):
    if isinstance(x, str):
        return list(ast.literal_eval(x))
    return list(x)

def answer_to_order(answer):
    return tuple(sorted([1, 2, 3, 4], key=lambda i: answer[i - 1]))

def order_to_answer(order):
    ans = [0, 0, 0, 0]
    for pos, img_num in enumerate(order, start=1):
        ans[img_num - 1] = pos
    return ans

def answer_to_class(answer):
    return PERM_TO_CLASS[answer_to_order(answer)]

def class_to_answer(cls):
    return order_to_answer(CLASS_TO_PERM[int(cls)])

def transform_label_by_input_perm(original_answer, input_perm, no_ordering=False):
    if no_ordering:
        return IDENTITY_CLASS

    original_order = answer_to_order(original_answer)
    cur_slot = {orig_img: new_slot + 1 for new_slot, orig_img in enumerate(input_perm)}
    new_order = tuple(cur_slot[orig_img] for orig_img in original_order)
    return PERM_TO_CLASS[new_order]

train_df["Answer_list"] = train_df["Answer"].apply(parse_answer)
train_df["label_cls"] = train_df["Answer_list"].apply(answer_to_class)
train_df["No_ordering"] = train_df["No_ordering"].astype(bool)

print("train:", train_df.shape)
print("test:", test_df.shape)
print("no_order ratio:", train_df["No_ordering"].mean())

train: (9535, 10)
test: (819, 6)
no_order ratio: 0.15500786575773468


In [18]:
# Cell 3. Train / Valid Split

try:
    from sklearn.model_selection import train_test_split

    train_part, valid_part = train_test_split(
        train_df,
        test_size=0.20,
        random_state=SEED,
        stratify=train_df["label_cls"],
    )
except Exception:
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(train_df))
    rng.shuffle(idx)
    valid_size = int(len(idx) * 0.20)
    valid_idx = idx[:valid_size]
    train_idx = idx[valid_size:]
    train_part = train_df.iloc[train_idx].copy()
    valid_part = train_df.iloc[valid_idx].copy()

train_part = train_part.reset_index(drop=True)
valid_part = valid_part.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)
print("valid no_order ratio:", valid_part["No_ordering"].mean())

train_part: (7628, 10)
valid_part: (1907, 10)
valid no_order ratio: 0.1546932354483482


In [19]:
# Cell 3. Train / Valid Split

try:
    from sklearn.model_selection import train_test_split

    train_part, valid_part = train_test_split(
        train_df,
        test_size=0.20,
        random_state=SEED,
        stratify=train_df["label_cls"],
    )
except Exception:
    rng = np.random.default_rng(SEED)
    idx = np.arange(len(train_df))
    rng.shuffle(idx)
    valid_size = int(len(idx) * 0.20)
    valid_idx = idx[:valid_size]
    train_idx = idx[valid_size:]
    train_part = train_df.iloc[train_idx].copy()
    valid_part = train_df.iloc[valid_idx].copy()

train_part = train_part.reset_index(drop=True)
valid_part = valid_part.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("train_part:", train_part.shape)
print("valid_part:", valid_part.shape)

train_part: (7628, 10)
valid_part: (1907, 10)


In [20]:
# Cell 4. Image Path / Text Views

def resolve_image_path(path_value, mode):
    p = Path(str(path_value))
    if p.exists():
        return str(p)

    base_dir = TEST_IMAGE_DIR if mode == "test" else TRAIN_IMAGE_DIR
    filename = p.name

    p1 = base_dir / filename
    if p1.exists():
        return str(p1)

    folder = filename.split("_")[0]
    p2 = base_dir / folder / filename
    if p2.exists():
        return str(p2)

    matches = list(base_dir.rglob(filename))
    if matches:
        return str(matches[0])

    raise FileNotFoundError(f"Image not found: {filename}")

def split_events(sentence):
    s = str(sentence).strip()
    parts = re.split(
        r"\b(?:then|after that|afterward|afterwards|followed by|before|while|and then|finally)\b|[.;]",
        s,
        flags=re.IGNORECASE,
    )
    parts = [p.strip() for p in parts if p.strip()]

    if len(parts) >= 2:
        early = parts[0]
        late = parts[-1]
    else:
        words = s.split()
        mid = max(1, len(words) // 2)
        early = " ".join(words[:mid])
        late = " ".join(words[mid:]) if len(words) > 1 else s

    return s, early, late

def make_text_views(sentence):
    full, early, late = split_events(sentence)
    return [
        full,
        "early event: " + early,
        "late event: " + late,
        "chronological order clue: " + full,
        "static or no clear ordering: " + full,
    ]

def get_image_paths(df, mode):
    return [
        [resolve_image_path(row[f"Input_{i}"], mode) for i in [1, 2, 3, 4]]
        for _, row in df.iterrows()
    ]

In [21]:
# Cell 5. Cached SigLIP2 Feature Extraction

CACHE_DIR = MODEL_SAVE_DIR / ("cache_" + MODEL_ID.replace("/", "_").replace("-", "_"))
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def as_feature_tensor(out):
    if torch.is_tensor(out):
        return out
    if hasattr(out, "pooler_output") and out.pooler_output is not None:
        return out.pooler_output
    if hasattr(out, "image_embeds"):
        return out.image_embeds
    if hasattr(out, "text_embeds"):
        return out.text_embeds
    if hasattr(out, "last_hidden_state"):
        return out.last_hidden_state[:, 0]
    raise TypeError(type(out))

def cached_build(name, builder):
    path = CACHE_DIR / f"{name}.pt"
    if path.exists():
        print("load cache:", path)
        return torch.load(path, map_location="cpu", weights_only=False)

    feats = builder()
    torch.save(feats, path)
    print("save cache:", path)
    return feats

if not all((CACHE_DIR / f"{x}.pt").exists() for x in ["train_feats", "valid_feats", "test_feats"]):
    from transformers import AutoModel, AutoProcessor

    processor = AutoProcessor.from_pretrained(MODEL_ID)
    clip_model = AutoModel.from_pretrained(MODEL_ID, torch_dtype=torch.bfloat16).to(device)
    clip_model.eval()

    @torch.no_grad()
    def encode_images(all_paths, desc):
        flat = []
        for paths in all_paths:
            flat.extend(paths)

        feats = []
        for i in tqdm(range(0, len(flat), IMAGE_BATCH_SIZE), desc=desc):
            paths = flat[i:i + IMAGE_BATCH_SIZE]
            images = [Image.open(p).convert("RGB") for p in paths]
            inputs = processor(images=images, return_tensors="pt").to(device)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
                if hasattr(clip_model, "get_image_features"):
                    out = clip_model.get_image_features(**inputs)
                else:
                    out = clip_model.vision_model(**inputs)

            feat = F.normalize(as_feature_tensor(out).float(), dim=-1)
            feats.append(feat.cpu())

        feats = torch.cat(feats, dim=0)
        return feats.view(len(all_paths), 4, -1)

    @torch.no_grad()
    def encode_texts(sentences, desc):
        texts = []
        for s in sentences:
            texts.extend(make_text_views(s))

        feats = []
        for i in tqdm(range(0, len(texts), TEXT_BATCH_SIZE), desc=desc):
            batch = texts[i:i + TEXT_BATCH_SIZE]

            inputs = processor(
                text=batch,
                return_tensors="pt",
                padding="max_length",
                truncation=True,
                max_length=64,
            ).to(device)

            with torch.amp.autocast("cuda", dtype=torch.bfloat16, enabled=(device.type == "cuda")):
                if hasattr(clip_model, "get_text_features"):
                    out = clip_model.get_text_features(**inputs)
                else:
                    out = clip_model.text_model(**inputs)

            feat = F.normalize(as_feature_tensor(out).float(), dim=-1)
            feats.append(feat.cpu())

        feats = torch.cat(feats, dim=0)
        return feats.view(len(sentences), 5, -1)

    def build_feats(df, mode, desc):
        image_feats = encode_images(get_image_paths(df, mode), f"{desc} images")
        text_feats = encode_texts(df["Sentence"].tolist(), f"{desc} texts")

        result = {
            "image_feats": image_feats,
            "text_feats": text_feats,
            "ids": df["Id"].tolist(),
        }

        if "Answer" in df.columns:
            answers = [parse_answer(x) for x in df["Answer"].tolist()]
            no_ordering = np.array(df["No_ordering"].astype(bool).tolist())

            result["answers"] = answers
            result["labels"] = torch.tensor([answer_to_class(a) for a in answers], dtype=torch.long)
            result["no_ordering"] = no_ordering
            result["weights"] = torch.tensor(
                [NO_ORDERING_WEIGHT if x else 1.0 for x in no_ordering],
                dtype=torch.float32,
            )

        return result

else:
    print("all feature cache exists:", CACHE_DIR)

train_feats = cached_build("train_feats", lambda: build_feats(train_part, "train", "train"))
valid_feats = cached_build("valid_feats", lambda: build_feats(valid_part, "train", "valid"))
test_feats = cached_build("test_feats", lambda: build_feats(test_df, "test", "test"))

print(train_feats["image_feats"].shape, train_feats["text_feats"].shape)
print(valid_feats["image_feats"].shape, valid_feats["text_feats"].shape)
print(test_feats["image_feats"].shape, test_feats["text_feats"].shape)

all feature cache exists: C:\SNU_2026\models\single_siglip2_high_score\cache_google_siglip2_so400m_patch16_384
load cache: C:\SNU_2026\models\single_siglip2_high_score\cache_google_siglip2_so400m_patch16_384\train_feats.pt
load cache: C:\SNU_2026\models\single_siglip2_high_score\cache_google_siglip2_so400m_patch16_384\valid_feats.pt
load cache: C:\SNU_2026\models\single_siglip2_high_score\cache_google_siglip2_so400m_patch16_384\test_feats.pt
torch.Size([7628, 4, 1152]) torch.Size([7628, 5, 1152])
torch.Size([1907, 4, 1152]) torch.Size([1907, 5, 1152])
torch.Size([819, 4, 1152]) torch.Size([819, 5, 1152])


In [22]:
# Cell 6. Pairwise Targets

pair_signs = []
pos_targets = []

for perm in PERMS:
    rank = {slot - 1: pos for pos, slot in enumerate(perm)}
    pair_signs.append([1.0 if rank[a] < rank[b] else -1.0 for a, b in PAIR_INDEX])

    p = [0, 0, 0, 0]
    for pos, slot in enumerate(perm):
        p[slot - 1] = pos
    pos_targets.append(p)

CLASS_PAIR_SIGNS = torch.tensor(pair_signs, dtype=torch.float32).to(device)
CLASS_PAIR_TARGETS = ((CLASS_PAIR_SIGNS + 1.0) / 2.0).to(device)
CLASS_POS_TARGETS = torch.tensor(pos_targets, dtype=torch.long).to(device)

In [23]:
# Cell 7. Dataset / Model

class PairwiseFeatureDataset(Dataset):
    def __init__(self, feats, training=False):
        self.img = feats["image_feats"].float()
        self.txt = feats["text_feats"].float()
        self.labels = feats.get("labels", None)
        self.answers = feats.get("answers", None)
        self.no_ordering = feats.get("no_ordering", np.zeros(len(self.img), dtype=bool))
        self.weights = feats.get("weights", torch.ones(len(self.img)))
        self.training = training

    def __len__(self):
        return len(self.img)

    def __getitem__(self, idx):
        img = self.img[idx]
        txt = self.txt[idx]

        if self.labels is None:
            label = torch.tensor(-1, dtype=torch.long)
            no_ordering = torch.tensor(0.0)
            weight = torch.tensor(1.0)
        else:
            no_ordering = torch.tensor(float(self.no_ordering[idx]))
            weight = self.weights[idx].float()

            if self.training:
                input_perm = random.choice(PERMS)
                img = img[[x - 1 for x in input_perm]]
                label = transform_label_by_input_perm(
                    self.answers[idx],
                    input_perm,
                    bool(no_ordering.item() > 0.5),
                )
                label = torch.tensor(label, dtype=torch.long)
            else:
                label = self.labels[idx].long()

        return {
            "image_feats": img,
            "text_feats": txt,
            "labels": label,
            "no_ordering": no_ordering,
            "weights": weight,
        }


class PairwiseTemporalRanker(nn.Module):
    def __init__(self, dim, n_text):
        super().__init__()

        pair_dim = 5 * dim + 2 * n_text + 1
        pos_dim = 4 * dim + n_text
        global_dim = 9 * dim + 6

        self.pair_mlp = nn.Sequential(
            nn.LayerNorm(pair_dim),
            nn.Linear(pair_dim, 768),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(768, 256),
            nn.GELU(),
            nn.Dropout(0.15),
            nn.Linear(256, 1),
        )

        self.pos_mlp = nn.Sequential(
            nn.LayerNorm(pos_dim),
            nn.Linear(pos_dim, 512),
            nn.GELU(),
            nn.Dropout(0.20),
            nn.Linear(512, 4),
        )

        self.noorder_mlp = nn.Sequential(
            nn.LayerNorm(global_dim),
            nn.Linear(global_dim, 512),
            nn.GELU(),
            nn.Dropout(0.25),
            nn.Linear(512, 1),
        )

    def forward(self, img, txt):
        txt_mean = txt.mean(dim=1)

        pair_logits = []
        for a, b in PAIR_INDEX:
            ia, ib = img[:, a], img[:, b]
            sa = torch.einsum("bd,btd->bt", ia, txt)
            sb = torch.einsum("bd,btd->bt", ib, txt)
            cos_ab = (ia * ib).sum(-1, keepdim=True)

            x = torch.cat(
                [ia, ib, ia - ib, ia * ib, torch.abs(ia - ib), sa, sb, cos_ab],
                dim=1,
            )
            pair_logits.append(self.pair_mlp(x).squeeze(1))

        pair_logits = torch.stack(pair_logits, dim=1)
        pair_scores = pair_logits @ CLASS_PAIR_SIGNS.T

        sims = torch.einsum("bkd,btd->bkt", img, txt)
        txt_rep = txt_mean[:, None, :].expand(-1, 4, -1)

        pos_x = torch.cat(
            [img, txt_rep, img * txt_rep, torch.abs(img - txt_rep), sims],
            dim=2,
        )
        pos_logits = self.pos_mlp(pos_x)

        pos_scores = []
        for perm in PERMS:
            s = 0
            for pos, slot in enumerate(perm):
                s = s + pos_logits[:, slot - 1, pos]
            pos_scores.append(s)
        pos_scores = torch.stack(pos_scores, dim=1)

        pair_cos = torch.stack(
            [(img[:, a] * img[:, b]).sum(-1) for a, b in PAIR_INDEX],
            dim=1,
        )
        global_x = torch.cat(
            [img.reshape(img.size(0), -1), txt.reshape(txt.size(0), -1), pair_cos],
            dim=1,
        )
        no_logits = self.noorder_mlp(global_x).squeeze(1)

        scores = pair_scores + 0.30 * pos_scores
        scores[:, IDENTITY_CLASS] += 0.10 * no_logits

        return scores, pair_logits, pos_logits, no_logits

In [24]:
# Cell 8. Train / Evaluate Functions

dim = train_feats["image_feats"].shape[-1]
n_text = train_feats["text_feats"].shape[1]

model = PairwiseTemporalRanker(dim, n_text).to(device)

labels_train = train_feats["labels"].long()
freq = torch.bincount(labels_train, minlength=24).float()
class_weights = (freq.sum() / freq.clamp_min(1)).pow(0.25)
class_weights = (class_weights / class_weights.mean()).to(device)

no_y = torch.tensor(np.asarray(train_feats["no_ordering"]).astype(np.float32))
pos_weight = torch.sqrt((len(no_y) - no_y.sum()) / no_y.sum()).to(device)

train_loader = DataLoader(
    PairwiseFeatureDataset(train_feats, training=True),
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    pin_memory=True,
)

valid_loader = DataLoader(
    PairwiseFeatureDataset(valid_feats, training=False),
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
)

optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

def move_batch(batch):
    return {k: v.to(device, non_blocking=True) if torch.is_tensor(v) else v for k, v in batch.items()}

def train_one_epoch(epoch):
    model.train()
    total_loss, total_correct, total_count = 0.0, 0, 0

    for batch in tqdm(train_loader, desc=f"epoch {epoch}"):
        batch = move_batch(batch)

        img = batch["image_feats"]
        txt = batch["text_feats"]
        labels = batch["labels"]
        no_ordering = batch["no_ordering"]
        weights = batch["weights"]

        scores, pair_logits, pos_logits, no_logits = model(img, txt)

        loss_cls_raw = F.cross_entropy(scores, labels, weight=class_weights, reduction="none")
        loss_cls = (loss_cls_raw * weights).mean()

        order_mask = no_ordering < 0.5

        if order_mask.any():
            pair_targets = CLASS_PAIR_TARGETS[labels]
            pos_targets = CLASS_POS_TARGETS[labels]

            loss_pair = F.binary_cross_entropy_with_logits(
                pair_logits[order_mask],
                pair_targets[order_mask],
            )

            loss_pos = F.cross_entropy(
                pos_logits[order_mask].reshape(-1, 4),
                pos_targets[order_mask].reshape(-1),
            )
        else:
            loss_pair = torch.tensor(0.0, device=device)
            loss_pos = torch.tensor(0.0, device=device)

        loss_no = F.binary_cross_entropy_with_logits(
            no_logits,
            no_ordering,
            pos_weight=pos_weight,
        )

        loss = loss_cls + 1.20 * loss_pair + 0.15 * loss_pos + 0.03 * loss_no

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()

        total_loss += loss.item() * labels.size(0)
        total_correct += (scores.argmax(dim=1) == labels).sum().item()
        total_count += labels.size(0)

    return total_loss / total_count, total_correct / total_count

@torch.no_grad()
def evaluate_collect(loader):
    model.eval()
    all_logits, all_labels = [], []

    for batch in tqdm(loader, desc="valid"):
        batch = move_batch(batch)
        scores, _, _, _ = model(batch["image_feats"], batch["text_feats"])
        all_logits.append(scores.float().cpu())
        all_labels.append(batch["labels"].cpu())

    return torch.cat(all_logits), torch.cat(all_labels)

def calibrate_identity_bias_and_threshold(logits, labels):
    best_acc = 0.0
    best_bias = 0.0
    best_thr = 0.0
    best_preds = None

    for bias in torch.linspace(0.0, 4.0, 81):
        scores = logits.clone()
        scores[:, IDENTITY_CLASS] += bias

        probs = scores.softmax(dim=1)
        top2 = probs.topk(2, dim=1).values
        margins = top2[:, 0] - top2[:, 1]

        for thr in torch.linspace(0.0, 0.35, 71):
            preds = scores.argmax(dim=1).clone()
            preds[margins < thr] = IDENTITY_CLASS

            acc = (preds == labels).float().mean().item()

            if acc > best_acc:
                best_acc = acc
                best_bias = float(bias)
                best_thr = float(thr)
                best_preds = preds.clone()

    return best_bias, best_thr, best_acc, best_preds

In [25]:
# Cell 9. Train Or Load Best Checkpoint

if USE_EXISTING_CHECKPOINT and BEST_PATH.exists():
    ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["head_state"])
    best_acc = ckpt["best_acc"]
    best_bias = ckpt["best_bias"]
    best_thr = ckpt["best_thr"]

    print("loaded:", BEST_PATH)
    print("best_acc:", best_acc)
    print("best_bias:", best_bias)
    print("best_thr:", best_thr)

else:
    best_acc = 0.0
    best_bias = 0.0
    best_thr = 0.0
    bad_epochs = 0

    for epoch in range(1, EPOCHS + 1):
        train_loss, train_acc = train_one_epoch(epoch)

        valid_logits, valid_labels = evaluate_collect(valid_loader)
        raw_acc = (valid_logits.argmax(dim=1) == valid_labels).float().mean().item()

        bias, thr, cal_acc, cal_preds = calibrate_identity_bias_and_threshold(
            valid_logits,
            valid_labels,
        )

        id_ratio = (cal_preds == IDENTITY_CLASS).float().mean().item()

        print(
            f"epoch {epoch:02d} | "
            f"loss {train_loss:.4f} | train {train_acc:.4f} | "
            f"raw {raw_acc:.4f} | cal {cal_acc:.4f} | "
            f"id_ratio {id_ratio:.3f} | bias {bias:.3f} | thr {thr:.3f}"
        )

        if cal_acc > best_acc:
            best_acc = cal_acc
            best_bias = bias
            best_thr = thr
            bad_epochs = 0

            torch.save(
                {
                    "head_state": model.state_dict(),
                    "best_acc": best_acc,
                    "best_epoch": epoch,
                    "best_bias": best_bias,
                    "best_thr": best_thr,
                    "dim": dim,
                    "n_text": n_text,
                },
                BEST_PATH,
            )
            print("saved:", BEST_PATH)
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print("early stop")
                break

    print("best_acc:", best_acc)

epoch 1:   0%|          | 0/30 [00:00<?, ?it/s]

valid:   0%|          | 0/8 [00:00<?, ?it/s]

epoch 01 | loss 3.9675 | train 0.1184 | raw 0.1730 | cal 0.1830 | id_ratio 0.856 | bias 0.400 | thr 0.000
saved: C:\SNU_2026\models\single_siglip2_high_score\best_pairwise_ranker.pt


epoch 2:   0%|          | 0/30 [00:00<?, ?it/s]

valid:   0%|          | 0/8 [00:00<?, ?it/s]

...

valid:   0%|          | 0/8 [00:00<?, ?it/s]

epoch 29 | loss 1.5002 | train 0.6818 | raw 0.3256 | cal 0.3471 | id_ratio 0.460 | bias 1.250 | thr 0.035


epoch 30:   0%|          | 0/30 [00:00<?, ?it/s]

valid:   0%|          | 0/8 [00:00<?, ?it/s]

epoch 30 | loss 1.4643 | train 0.6850 | raw 0.3571 | cal 0.3776 | id_ratio 0.419 | bias 1.300 | thr 0.060
saved: C:\SNU_2026\models\single_siglip2_high_score\best_pairwise_ranker.pt
best_acc: 0.37755638360977173


In [26]:
# Cell 10. TTA Predict

ckpt = torch.load(BEST_PATH, map_location=device, weights_only=False)
model = PairwiseTemporalRanker(ckpt["dim"], ckpt["n_text"]).to(device)
model.load_state_dict(ckpt["head_state"])
model.eval()

def map_probs_back(probs, input_perm):
    mapped = torch.zeros_like(probs)

    for new_cls, new_order in CLASS_TO_PERM.items():
        original_order = tuple(input_perm[new_slot - 1] for new_slot in new_order)
        original_cls = PERM_TO_CLASS[original_order]
        mapped[:, original_cls] += probs[:, new_cls]

    return mapped

@torch.no_grad()
def predict_tta(feats, tta_perms=PERMS):
    img_all = feats["image_feats"].float()
    txt_all = feats["text_feats"].float()

    total_probs = torch.zeros(len(img_all), 24)

    for input_perm in tqdm(tta_perms, desc="TTA"):
        idx = torch.tensor([x - 1 for x in input_perm], dtype=torch.long)

        probs_list = []

        for start in range(0, len(img_all), BATCH_SIZE):
            end = start + BATCH_SIZE

            img = img_all[start:end][:, idx].to(device)
            txt = txt_all[start:end].to(device)

            scores, _, _, _ = model(img, txt)
            probs = scores.softmax(dim=1).float().cpu()

            probs_list.append(map_probs_back(probs, input_perm))

        total_probs += torch.cat(probs_list)

    avg_probs = total_probs / len(tta_perms)
    return torch.log(avg_probs.clamp_min(1e-8))

def apply_bias_threshold(logits, bias, thr):
    scores = logits.clone()
    scores[:, IDENTITY_CLASS] += bias

    probs = scores.softmax(dim=1)
    top2 = probs.topk(2, dim=1).values
    margins = top2[:, 0] - top2[:, 1]

    preds = scores.argmax(dim=1).clone()
    preds[margins < thr] = IDENTITY_CLASS

    return preds

In [27]:
# Cell 11. Greedy TTA Selection

@torch.no_grad()
def predict_each_tta_perm(feats, tta_perms=PERMS):
    img_all = feats["image_feats"].float()
    txt_all = feats["text_feats"].float()

    all_probs = []

    for input_perm in tqdm(tta_perms, desc="each TTA perm"):
        idx = torch.tensor([x - 1 for x in input_perm], dtype=torch.long)
        probs_list = []

        for start in range(0, len(img_all), BATCH_SIZE):
            end = start + BATCH_SIZE

            img = img_all[start:end][:, idx].to(device)
            txt = txt_all[start:end].to(device)

            scores, _, _, _ = model(img, txt)
            probs = scores.softmax(dim=1).float().cpu()
            probs = map_probs_back(probs, input_perm)

            probs_list.append(probs)

        all_probs.append(torch.cat(probs_list))

    return torch.stack(all_probs, dim=0)


def probs_to_logits(probs):
    return torch.log(probs.clamp_min(1e-8))


valid_probs_each = predict_each_tta_perm(valid_feats, PERMS)
valid_labels = valid_feats["labels"]

best_subset = []
remaining = list(range(len(PERMS)))

best_acc = -1
best_bias = 0
best_thr = 0

for step in range(1, 13):  # 24개 전부 말고 최대 12개까지만 선택
    step_best = None

    for cand in remaining:
        subset = best_subset + [cand]
        probs = valid_probs_each[subset].mean(dim=0)
        logits = probs_to_logits(probs)

        bias, thr, acc, preds = calibrate_identity_bias_and_threshold(
            logits,
            valid_labels,
        )

        id_ratio = (preds == IDENTITY_CLASS).float().mean().item()

        # identity collapse 방지
        if id_ratio > 0.36:
            continue

        if step_best is None or acc > step_best["acc"]:
            step_best = {
                "cand": cand,
                "acc": acc,
                "bias": bias,
                "thr": thr,
                "id_ratio": id_ratio,
                "subset": subset,
            }

    if step_best is None:
        break

    if step_best["acc"] <= best_acc:
        break

    best_subset = step_best["subset"]
    remaining.remove(step_best["cand"])

    best_acc = step_best["acc"]
    best_bias = step_best["bias"]
    best_thr = step_best["thr"]

    print(
        f"step {step:02d} | "
        f"acc {best_acc:.4f} | "
        f"id_ratio {step_best['id_ratio']:.3f} | "
        f"add_perm {PERMS[step_best['cand']]} | "
        f"bias {best_bias:.3f} | thr {best_thr:.3f}"
    )

selected_tta_perms = [PERMS[i] for i in best_subset]

print("selected_tta_count:", len(selected_tta_perms))
print("selected_tta_perms:", selected_tta_perms)
print("best greedy TTA valid_acc:", best_acc)
print("best_bias:", best_bias)
print("best_thr:", best_thr)

each TTA perm:   0%|          | 0/24 [00:00<?, ?it/s]

step 01 | acc 0.3928 | id_ratio 0.346 | add_perm (2, 3, 4, 1) | bias 1.150 | thr 0.055
step 02 | acc 0.4164 | id_ratio 0.306 | add_perm (4, 1, 3, 2) | bias 0.950 | thr 0.030
step 03 | acc 0.4190 | id_ratio 0.339 | add_perm (3, 4, 2, 1) | bias 0.950 | thr 0.035
step 04 | acc 0.4248 | id_ratio 0.299 | add_perm (1, 2, 4, 3) | bias 1.200 | thr 0.015
selected_tta_count: 4
selected_tta_perms: [(2, 3, 4, 1), (4, 1, 3, 2), (3, 4, 2, 1), (1, 2, 4, 3)]
best greedy TTA valid_acc: 0.4247509241104126
best_bias: 1.2000000476837158
best_thr: 0.014999999664723873


In [28]:
# Cell 12. Submission With Selected TTA

test_logits_tta = predict_tta(test_feats, selected_tta_perms)
test_preds = apply_bias_threshold(test_logits_tta, best_bias, best_thr)

submission = pd.DataFrame({
    "Id": test_feats["ids"],
    "Answer": [str(class_to_answer(int(x))) for x in test_preds],
})

submission.to_csv(OUTPUT_PATH, index=False)

print("saved:", OUTPUT_PATH)
print("submission shape:", submission.shape)
print("test identity_ratio:", (test_preds == IDENTITY_CLASS).float().mean().item())
submission.head()

TTA:   0%|          | 0/4 [00:00<?, ?it/s]

saved: C:\SNU_2026\submission_pairwise_ranker_safe.csv
submission shape: (819, 2)
test identity_ratio: 0.27716729044914246


,Id,Answer
0,WI8o3F,"[1, 4, 2, 3]"
1,Ae7zLm,"[3, 1, 4, 2]"
2,4QSIII,"[2, 4, 1, 3]"
3,k7h4gt,"[4, 1, 2, 3]"
4,ZtTwCx,"[1, 2, 3, 4]"
